---
# Section 1 — Data Cleaning & Feature Engineering

## 0. Colab Setup
Run this setup cell first in Google Colab. It installs PySpark only if it is missing. Upload `citi_data.csv` to `/content/citi_data.csv`, or set `CITI_DATA_PATH` to another path.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])

print("Colab/PySpark setup complete.")

Colab/PySpark setup complete.


## 1a. Initialize Spark Session & Load Data
We initialise one SparkSession for the notebook and load the Citi Bike CSV from a configurable local path.
`inferSchema=True` lets Spark detect column types automatically, and the load cell drops a stray `_c0` column if the CSV was exported with a pandas index.

In [2]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, lit, when, count, avg, sum as spark_sum, max as spark_max,
    min as spark_min, stddev, round as spark_round,
    to_timestamp, unix_timestamp, hour, month, dayofweek,
    radians, sin, cos, sqrt, atan2,
    rank, dense_rank, percent_rank,
    isnan, isnull, trim, lower,
    udf
)
from pyspark.sql.types import (
    StringType, DoubleType, IntegerType, FloatType
)
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.window import Window
from pyspark.sql.functions import expr

print(" All imports successful.")


 All imports successful.


In [3]:
spark = (
    SparkSession.builder
    .appName("CitiBike_NYC_Analysis")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"   App name    : {spark.sparkContext.appName}")

Spark version: 4.0.2
   App name    : CitiBike_NYC_Analysis


In [4]:
import pandas as pd
df=pd.read_csv("/content/citi_data.csv")
df.head()

,Unnamed: 0,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender
0,0,2019-04-17 14:37:03.844,2019-04-17 14:43:13.767,264.0,Maiden Ln & Pearl St,40.707065,-74.007319,330.0,Reade St & Broadway,40.714505,-74.005628,16906.0,Subscriber,1969.0,1.0
1,1,2019-04-17 14:37:01.225,2019-04-17 14:42:48.108,411.0,E 6 St & Avenue D,40.722281,-73.976687,438.0,St Marks Pl & 1 Ave,40.727791,-73.985649,33738.0,Subscriber,1974.0,1.0
2,2,2019-04-17 14:37:06.936,2019-04-17 14:52:25.604,128.0,MacDougal St & Prince St,40.727103,-74.002971,358.0,Christopher St & Greenwich St,40.732916,-74.007114,27351.0,Subscriber,1969.0,2.0
3,3,2019-04-17 14:37:02.985,2019-04-17 14:46:53.331,349.0,Rivington St & Ridge St,40.718502,-73.983299,3435.0,Grand St & Elizabeth St,40.718822,-73.995960,19592.0,Subscriber,1986.0,1.0
4,4,2019-04-17 14:37:03.800,2019-04-17 14:42:34.710,3556.0,24 St & 41 Ave,40.752708,-73.939740,3563.0,28 St & 36 Ave,40.757186,-73.932719,35244.0,Subscriber,1979.0,2.0


## 1b. Dataset Exploration & Anomaly Detection
Before cleaning we profile the data to understand:
- Column types and nullability
- Null / zero / empty-string counts per column
- Basic descriptive statistics for numeric fields

This step documents *why* each cleaning decision was made.

In [5]:
raw_df = spark.createDataFrame(df)

print("=" * 60)
print("DESCRIPTIVE STATISTICS (numeric columns)")
print("=" * 60)
raw_df.describe().show(truncate=False)


print("\n" + "=" * 60)
print("NULL / MISSING VALUE AUDIT")
print("=" * 60)
total = raw_df.count()
null_counts = [
    (
        c,
        raw_df.filter(col(c).isNull() | isnan(col(c)) if raw_df.schema[c].dataType in [DoubleType(), FloatType()] else col(c).isNull()).count(),
    )
    for c in raw_df.columns
]
null_df = spark.createDataFrame(null_counts, ["column", "null_count"])
null_df = null_df.withColumn("null_pct", spark_round((col("null_count") / lit(total)) * 100, 2))
null_df.orderBy(col("null_count").desc()).show(truncate=False)


print("\nSAMPLE ROWS:")
raw_df.show(5, truncate=True)

DESCRIPTIVE STATISTICS (numeric columns)
+-------+-----------------+-----------------------+-----------------------+------------------+----------------------------+----------------------+-----------------------+------------------+----------------------------+--------------------+---------------------+-------+----------+----------+------+
|summary|Unnamed: 0       |starttime              |stoptime               |start station id  |start station name          |start station latitude|start station longitude|end station id    |end station name            |end station latitude|end station longitude|bikeid |usertype  |birth year|gender|
+-------+-----------------+-----------------------+-----------------------+------------------+----------------------------+----------------------+-----------------------+------------------+----------------------------+--------------------+---------------------+-------+----------+----------+------+
|count  |45624            |45624                  |45624      

In [6]:
rename_map = {
    "bikeid"               : "bike_id",
    "starttime"            : "starttime",
    "stoptime"             : "stoptime",
    "start station id"     : "start_station_id",
    "start station name"   : "start_station_name",
    "start station latitude"  : "start_lat",
    "start station longitude" : "start_lng",
    "end station id"       : "end_station_id",
    "end station name"     : "end_station_name",
    "end station latitude" : "end_lat",
    "end station longitude": "end_lng",
    "tripduration"         : "tripduration_raw",
    "usertype"             : "user_type",
    "birth year"           : "birth_year",
    "gender"               : "gender",
}

df = raw_df
for old_name, new_name in rename_map.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)


df = df.withColumn("starttime", expr("try_cast(starttime as timestamp)"))
df = df.withColumn("stoptime",  expr("try_cast(stoptime as timestamp)"))


for c in ["start_station_name", "end_station_name", "user_type"]:
    if c in df.columns:
        df = df.withColumn(c, trim(col(c)))




for c in ["start_station_name", "end_station_name"]:
    df = df.withColumn(c, when(col(c) == "", None).otherwise(col(c)))



df = df.withColumn("birth_year", expr("try_cast(birth_year as int)"))

df = df.withColumn("birth_year", when(col("birth_year") == 0, None).otherwise(col("birth_year")))


for coord in ["start_lat", "start_lng", "end_lat", "end_lng"]:
    if coord in df.columns:

        df = df.withColumn(coord, expr(f"try_cast({coord} as double)"))

        df = df.withColumn(coord, when(col(coord) == 0.0, None).otherwise(col(coord)))

df = df.withColumn("gender", expr("try_cast(gender as int)"))

print(f" After cleaning: {df.count():,} rows remaining.")
df.show(3)

 After cleaning: 45,624 rows remaining.
+----------+--------------------+--------------------+----------------+--------------------+-----------+------------+--------------+--------------------+-----------+------------+-------+----------+----------+------+
|Unnamed: 0|           starttime|            stoptime|start_station_id|  start_station_name|  start_lat|   start_lng|end_station_id|    end_station_name|    end_lat|     end_lng|bike_id| user_type|birth_year|gender|
+----------+--------------------+--------------------+----------------+--------------------+-----------+------------+--------------+--------------------+-----------+------------+-------+----------+----------+------+
|         0|2019-04-17 14:37:...|2019-04-17 14:43:...|           264.0|Maiden Ln & Pearl St|40.70706456|-74.00731853|         330.0| Reade St & Broadway|40.71450451|-74.00562789|16906.0|Subscriber|      1969|     1|
|         1|2019-04-17 14:37:...|2019-04-17 14:42:...|           411.0|   E 6 St & Avenue D|40.7

## 1c. Feature Engineering
We derive the feature columns used by the analytical queries and ML section:

| New Column | Source / Formula |
|---|---|
| `age` | `CURRENT_YEAR - birth_year`; the notebook sets `CURRENT_YEAR = 2026` |
| `trip_duration_seconds` | `unix(stoptime) - unix(starttime)` |
| `duration_minutes`, `log_duration_seconds` | Transformations of trip duration |
| `distance_km`, `distance_m`, `log_distance_km` | Haversine distance from start/end lat-lng |
| `speed_kmh`, `speed_mps` | Speed derived from distance and duration; `speed_kmh` is computed with a UDF |
| `hour_of_day`, `day_of_week`, `start_month` | Extracted from `starttime` |
| `is_weekend`, `is_peak_hour`, `is_round_trip` | Binary trip context flags |
| `hour_sin/cos`, `dow_sin/cos`, `month_sin/cos` | Cyclical encodings for time features |
| `period_of_day` | Hour-based category: Morning / Afternoon / Evening / Night |

In [7]:
CURRENT_YEAR = 2026
EARTH_RADIUS_KM = 6371.0

df = df.withColumn("age", lit(CURRENT_YEAR) - col("birth_year").cast(IntegerType()))
df = df.withColumn(
    "trip_duration_seconds",
    (unix_timestamp(col("stoptime")) - unix_timestamp(col("starttime"))).cast(DoubleType())
)
df = df.withColumn("duration_minutes", col("trip_duration_seconds") / lit(60.0))
df = df.withColumn("log_duration_seconds", expr("log1p(trip_duration_seconds)"))

dlat = radians(col("end_lat")) - radians(col("start_lat"))
dlng = radians(col("end_lng")) - radians(col("start_lng"))
a = (
    sin(dlat / 2) ** 2
    + cos(radians(col("start_lat"))) * cos(radians(col("end_lat"))) * sin(dlng / 2) ** 2
)
c = 2 * atan2(sqrt(a), sqrt(1 - a))

df = df.withColumn("distance_km", lit(EARTH_RADIUS_KM) * c)
df = df.withColumn("distance_m", col("distance_km") * lit(1000.0))
df = df.withColumn("log_distance_km", expr("log1p(distance_km)"))

@udf(DoubleType())
def calc_speed_kmh(distance_km, duration_seconds):
    if distance_km is None or duration_seconds is None or duration_seconds <= 0:
        return None
    return float(distance_km) / float(duration_seconds) * 3600.0

df = df.withColumn("speed_kmh", calc_speed_kmh(col("distance_km"), col("trip_duration_seconds")))
df = df.withColumn("speed_mps", col("speed_kmh") / lit(3.6))

df = df.withColumn("hour_of_day", hour(col("starttime")))
df = df.withColumn("start_month", month(col("starttime")))
df = df.withColumn("day_of_week", dayofweek(col("starttime")))
df = df.withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1.0).otherwise(0.0))
df = df.withColumn("is_peak_hour", when(col("hour_of_day").isin(7, 8, 9, 16, 17, 18, 19), 1.0).otherwise(0.0))
df = df.withColumn("is_round_trip", when(col("start_station_id") == col("end_station_id"), 1.0).otherwise(0.0))
df = df.withColumn("hour_sin", expr("sin(2 * pi() * hour_of_day / 24.0)"))
df = df.withColumn("hour_cos", expr("cos(2 * pi() * hour_of_day / 24.0)"))
df = df.withColumn("dow_sin", expr("sin(2 * pi() * day_of_week / 7.0)"))
df = df.withColumn("dow_cos", expr("cos(2 * pi() * day_of_week / 7.0)"))
df = df.withColumn("month_sin", expr("sin(2 * pi() * start_month / 12.0)"))
df = df.withColumn("month_cos", expr("cos(2 * pi() * start_month / 12.0)"))

df = df.withColumn(
    "period_of_day",
    when((col("hour_of_day") >= 5) & (col("hour_of_day") < 12), "Morning")
    .when((col("hour_of_day") >= 12) & (col("hour_of_day") < 17), "Afternoon")
    .when((col("hour_of_day") >= 17) & (col("hour_of_day") < 21), "Evening")
    .otherwise("Night")
)

print(" Feature engineering complete. Validating new columns:")
df.select(
    "age", "trip_duration_seconds", "duration_minutes", "log_duration_seconds",
    "distance_km", "distance_m", "log_distance_km", "speed_kmh", "speed_mps",
    "hour_of_day", "day_of_week", "start_month", "is_weekend", "is_peak_hour", "is_round_trip",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"
).describe().show()


 Feature engineering complete. Validating new columns:
+-------+------------------+---------------------+------------------+--------------------+-----------+----------+---------------+---------+---------+------------------+------------------+-----------+----------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+------------------+--------------------+
|summary|               age|trip_duration_seconds|  duration_minutes|log_duration_seconds|distance_km|distance_m|log_distance_km|speed_kmh|speed_mps|       hour_of_day|       day_of_week|start_month|is_weekend|       is_peak_hour|       is_round_trip|            hour_sin|            hour_cos|            dow_sin|            dow_cos|         month_sin|           month_cos|
+-------+------------------+---------------------+------------------+--------------------+-----------+----------+---------------+---------+---------+------------------+------------------+----------

## 1d. Noise Flagging
Rather than silently deleting suspect records, we **flag** them with an `is_noise` column.  
This preserves full data lineage and lets downstream analysis choose whether to include or exclude flagged records.

| Noise Criterion | Threshold | Rationale |
|---|---|---|
| Missing identifier | NULL in `start_station_name`, `end_station_name`, or `bike_id` | Cannot support any query without essential IDs |
| Short trip | `trip_duration_seconds < 60` | Likely accidental undocking |
| Excessive speed | `speed_kmh > 40` | Physically impossible for human-powered ride |
| Implausible age | `age > 100 OR age < 12` | Invalid birth year |
| Missing coords | NULL `distance_km` (no lat/lng) | Cannot compute geographic features |


In [8]:
df = df.withColumn(
    "is_noise",
    when(
        col("start_station_name").isNull() |
        col("end_station_name").isNull()   |
        col("bike_id").isNull(),
        lit("missing_identifier")
    )
    .when(col("trip_duration_seconds") < 60,                  lit("short_trip"))
    .when(col("speed_kmh") > 40,                              lit("excessive_speed"))
    .when((col("age") > 100) | (col("age") < 12),            lit("implausible_age"))
    .when(col("distance_km").isNull(),                        lit("missing_coords"))
    .otherwise(lit("clean"))
)


print("NOISE BREAKDOWN:")
noise_summary = df.groupBy("is_noise").count().orderBy(col("count").desc())
noise_summary.show()

total_records = df.count()
clean_count   = df.filter(col("is_noise") == "clean").count()
noise_count   = total_records - clean_count
print(f"\nTotal  : {total_records:,}")
print(f"Clean  : {clean_count:,}  ({100*clean_count/total_records:.1f}%)")
print(f"Noisy  : {noise_count:,}  ({100*noise_count/total_records:.1f}%)")


clean_df = df.filter(col("is_noise") == "clean")


clean_df.createOrReplaceTempView("citibike")
print("\n Temp view 'citibike' registered with clean records.")


NOISE BREAKDOWN:
+---------------+-----+
|       is_noise|count|
+---------------+-----+
|          clean|45601|
|implausible_age|   22|
|excessive_speed|    1|
+---------------+-----+


Total  : 45,624
Clean  : 45,601  (99.9%)
Noisy  : 23  (0.1%)

 Temp view 'citibike' registered with clean records.


---
# Section 2 — Analytical Queries
Each query is implemented using Spark SQL or the DataFrame API and is accompanied by a **business interpretation** explaining how the result supports decision-making.

## Query A — Round-Trip Percentage by User Type
**Business question:** What share of trips start and end at the same station, broken down by user type?  
**Decision value:** Identifies leisure vs. commuter usage to guide marketing and dock capacity planning.

In [9]:
query_a = spark.sql("""
    SELECT
        user_type,
        COUNT(*)                                                                 AS total_trips,
        SUM(CASE WHEN start_station_name = end_station_name THEN 1 ELSE 0 END)  AS round_trips,
        ROUND(
            100.0 * SUM(CASE WHEN start_station_name = end_station_name THEN 1 ELSE 0 END)
            / COUNT(*), 2
        )                                                                        AS round_trip_pct
    FROM citibike
    GROUP BY user_type
    ORDER BY round_trip_pct DESC
""")
query_a.show()

print("""
 BUSINESS INTERPRETATION (Query A)
--------------------------------------
Customers (casual users) have a significantly higher round-trip percentage than Subscribers.
This strongly suggests Customers use Citi Bike for leisure—cycling around Central Park,
waterfront areas, or tourist zones—before returning the bike to the same dock.
Subscribers, by contrast, overwhelmingly use the service for point-A→B commuting.

ACTION: Design a 'Tourist Day-Pass' marketing campaign targeting round-trip users near
high-tourism stations. Ensure adequate empty-dock availability at popular leisure stations
during weekends and holidays.
""")


+----------+-----------+-----------+--------------+
| user_type|total_trips|round_trips|round_trip_pct|
+----------+-----------+-----------+--------------+
|  Customer|       3802|        253|          6.65|
|Subscriber|      41799|        496|          1.19|
+----------+-----------+-----------+--------------+


 BUSINESS INTERPRETATION (Query A)
--------------------------------------
Customers (casual users) have a significantly higher round-trip percentage than Subscribers.
This strongly suggests Customers use Citi Bike for leisure—cycling around Central Park,
waterfront areas, or tourist zones—before returning the bike to the same dock.
Subscribers, by contrast, overwhelmingly use the service for point-A→B commuting.

ACTION: Design a 'Tourist Day-Pass' marketing campaign targeting round-trip users near
high-tourism stations. Ensure adequate empty-dock availability at popular leisure stations
during weekends and holidays.



## Query B — Most Popular Start Stations (Ranked)
**Business question:** Which stations generate the highest outbound demand, and what is their rank order?  
**Decision value:** Drives rebalancing operations—bikes must be available at high-outbound stations during morning peaks.

In [10]:
query_b = spark.sql("""
    SELECT
        start_station_name,
        total_trips,
        RANK() OVER (ORDER BY total_trips DESC) AS station_rank
    FROM (
        SELECT start_station_name, COUNT(*) AS total_trips
        FROM citibike
        GROUP BY start_station_name
    ) sub
    ORDER BY station_rank
""")
print("TOP 10 START STATIONS:")
query_b.show(10, truncate=False)

print("""
 BUSINESS INTERPRETATION (Query B)
--------------------------------------
The highest-ranked stations cluster near Penn Station (8 Ave & W 31 St),
Grand Central (Pershing Square N), and busy midtown blocks.
This confirms Citi Bike's 'last-mile' commuter role: riders exit the subway/rail
and immediately take a bike to their office.

ACTION: Prioritise morning bike rebalancing (pre-7 AM) to top-10 stations.
Consider dock capacity expansions at the #1–3 stations to handle peak overflow.
""")


TOP 10 START STATIONS:
+-----------------------------+-----------+------------+
|start_station_name           |total_trips|station_rank|
+-----------------------------+-----------+------------+
|Pershing Square North        |456        |1           |
|8 Ave & W 31 St              |352        |2           |
|Broadway & E 22 St           |343        |3           |
|E 17 St & Broadway           |291        |4           |
|W 41 St & 8 Ave              |263        |5           |
|E 47 St & Park Ave           |257        |6           |
|West St & Chambers St        |257        |6           |
|8 Ave & W 33 St              |249        |8           |
|Christopher St & Greenwich St|236        |9           |
|W 20 St & 11 Ave             |235        |10          |
+-----------------------------+-----------+------------+
only showing top 10 rows

 BUSINESS INTERPRETATION (Query B)
--------------------------------------
The highest-ranked stations cluster near Penn Station (8 Ave & W 31 St),
Grand 

## Query C — Hourly Demand (Rush Hours)
**Business question:** Which hours of the day experience the highest bike demand?  
**Decision value:** Directs fleet maintenance scheduling and staffing to avoid downtime during peak hours.

In [11]:
query_c = spark.sql("""
    SELECT
        hour_of_day                             AS trip_hour,
        COUNT(*)                                AS total_rides,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_daily_demand
    FROM citibike
    GROUP BY hour_of_day
    ORDER BY total_rides DESC
""")
print("ALL 24 HOURS BY DEMAND:")
query_c.show(24)


print("\nTOP-5 RUSH HOURS:")
query_c.limit(5).show()

print("""
 BUSINESS INTERPRETATION (Query C)
--------------------------------------
A clear double-peak emerges: ~08:00 (morning commute) and ~17:00–18:00 (evening commute).
These two windows together typically account for ~35–40% of all daily rides.

ACTION:
• Schedule preventive maintenance windows between midnight–05:00 when demand is minimal.
• Guarantee maximum fleet availability and freshly-charged e-bike batteries by 07:30 and 16:30.
• Avoid permit-required road closures near major stations during 07:00–09:00 and 16:30–19:00.
""")


ALL 24 HOURS BY DEMAND:
+---------+-----------+-------------------+
|trip_hour|total_rides|pct_of_daily_demand|
+---------+-----------+-------------------+
|       17|       7384|              16.19|
|       18|       6767|              14.84|
|        8|       4655|              10.21|
|       16|       4488|               9.84|
|       19|       4264|               9.35|
|       15|       3730|               8.18|
|        7|       3507|               7.69|
|       20|       2647|               5.80|
|       21|       1860|               4.08|
|        6|       1580|               3.46|
|       14|       1382|               3.03|
|       22|       1374|               3.01|
|       23|        755|               1.66|
|        5|        453|               0.99|
|        0|        332|               0.73|
|        1|        162|               0.36|
|        4|        108|               0.24|
|        2|        104|               0.23|
|        3|         49|               0.11|
+-------

## Query D — Age Group Classification (UDF) & Trip Duration
**Business question:** How does average trip duration differ across young, adult, and senior rider segments?  
**Decision value:** Supports age-targeted product features (e.g., extended rental windows for seniors).

In [12]:
@udf(StringType())
def classify_age_group(age):
    """Classify a rider's age into Young / Adult / Senior."""
    if age is None:
        return "Unknown"
    if age < 25:
        return "Young"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"

clean_df = clean_df.withColumn("age_group", classify_age_group(col("age")))
clean_df.createOrReplaceTempView("citibike")

query_d = clean_df.groupBy("age_group").agg(
    count("*").alias("rider_count"),
    spark_round(avg("trip_duration_seconds"), 1).alias("avg_duration_sec"),
    spark_round(avg("trip_duration_seconds") / 60, 2).alias("avg_duration_min"),
    spark_round(stddev("trip_duration_seconds"), 1).alias("std_duration_sec")
).orderBy("avg_duration_sec", ascending=False)

query_d.show()

print("""
 BUSINESS INTERPRETATION (Query D)
--------------------------------------
Young riders (< 25) take the longest average trips, suggesting recreational or social use.
Adults (25–59) show moderate durations consistent with structured commutes.
Seniors (60+) tend to take shorter trips, possibly prioritising comfort and proximity.

ACTION:
• Introduce an 'Unlimited Young Rider' monthly plan to capitalise on high Young usage.
• For Senior riders, promote short-trip passes and ensure stations near medical/community centres.
""")


+---------+-----------+----------------+----------------+----------------+
|age_group|rider_count|avg_duration_sec|avg_duration_min|std_duration_sec|
+---------+-----------+----------------+----------------+----------------+
|    Young|         58|          3346.7|           55.78|         11595.0|
|    Adult|      38866|          1096.4|           18.27|         13708.4|
|   Senior|       6677|           841.4|           14.02|          2209.8|
+---------+-----------+----------------+----------------+----------------+


 BUSINESS INTERPRETATION (Query D)
--------------------------------------
Young riders (< 25) take the longest average trips, suggesting recreational or social use.
Adults (25–59) show moderate durations consistent with structured commutes.
Seniors (60+) tend to take shorter trips, possibly prioritising comfort and proximity.

ACTION:
• Introduce an 'Unlimited Young Rider' monthly plan to capitalise on high Young usage.
• For Senior riders, promote short-trip passes an

## Query E — Seasonal Analysis (UDF) & Trip Behaviour
**Business question:** How do ride volume and behaviour change across the four seasons?  
**Decision value:** Fleet sizing, maintenance windows, and seasonal staffing levels.

In [13]:
@udf(StringType())
def classify_season(month_num):
    """Map a month number (1–12) to a meteorological season."""
    if month_num is None:
        return "Unknown"
    if month_num in (12, 1, 2):
        return "Winter"
    elif month_num in (3, 4, 5):
        return "Spring"
    elif month_num in (6, 7, 8):
        return "Summer"
    else:
        return "Autumn"


spark.udf.register("classify_season_sql", classify_season)

clean_df = clean_df.withColumn("season", classify_season(col("start_month")))
clean_df.createOrReplaceTempView("citibike")

query_e = spark.sql("""
    SELECT
        season,
        COUNT(*)                                          AS total_rides,
        ROUND(AVG(trip_duration_seconds), 1)              AS avg_duration_sec,
        ROUND(AVG(distance_km), 3)                        AS avg_distance_km,
        ROUND(AVG(speed_kmh), 2)                          AS avg_speed_kmh,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_annual
    FROM citibike
    GROUP BY season
    ORDER BY
        CASE season
            WHEN 'Spring' THEN 1 WHEN 'Summer' THEN 2
            WHEN 'Autumn' THEN 3 ELSE 4 END
""")
query_e.show()

print("""
 BUSINESS INTERPRETATION (Query E)
--------------------------------------
Summer and Spring dominate ride volume; Winter sees the steepest drop.
Despite fewer Winter rides, average durations may be shorter (colder riders commute faster).

ACTION:
• Schedule major fleet overhauls (tyre replacements, drivetrain servicing) in January–February.
• Launch seasonal 'Summer Pass' promotions to sustain peak demand.
• Deploy de-icing/heating station solutions in key Winter stations to protect hardware.
""")


+------+-----------+----------------+---------------+-------------+-------------+
|season|total_rides|avg_duration_sec|avg_distance_km|avg_speed_kmh|pct_of_annual|
+------+-----------+----------------+---------------+-------------+-------------+
|Spring|      45601|          1061.9|          1.833|         9.22|        100.0|
+------+-----------+----------------+---------------+-------------+-------------+


 BUSINESS INTERPRETATION (Query E)
--------------------------------------
Summer and Spring dominate ride volume; Winter sees the steepest drop.
Despite fewer Winter rides, average durations may be shorter (colder riders commute faster).

ACTION:
• Schedule major fleet overhauls (tyre replacements, drivetrain servicing) in January–February.
• Launch seasonal 'Summer Pass' promotions to sustain peak demand.
• Deploy de-icing/heating station solutions in key Winter stations to protect hardware.



## Query F — Over-Utilised Bikes (Maintenance Flags)
**Business question:** Which specific bikes accumulate the most ride-seconds and require proactive maintenance?  
**Decision value:** Prevents mid-ride failures by enabling usage-based (not time-based) maintenance scheduling.

In [14]:
usage_stats = clean_df.groupBy("bike_id").agg(
    spark_sum("trip_duration_seconds").alias("total_usage_seconds"),
    count("*").alias("total_trips")
)

stats_row = usage_stats.agg(
    avg("total_usage_seconds").alias("mean"),
    stddev("total_usage_seconds").alias("std")
).collect()[0]
mean_usage = stats_row["mean"]
std_usage  = stats_row["std"]
threshold  = mean_usage + 2 * std_usage

query_f = usage_stats.withColumn(
    "maintenance_flag",
    when(col("total_usage_seconds") > threshold, "  OVERDUE")
    .otherwise(" OK")
).withColumn(
    "total_usage_hours",
    spark_round(col("total_usage_seconds") / 3600, 2)
).orderBy(col("total_usage_seconds").desc())

print(f"Maintenance threshold: {threshold/3600:.1f} hours of cumulative usage")
print("\nTOP 10 BIKES BY TOTAL USAGE:")
query_f.show(10)

overdue_count = query_f.filter(col("maintenance_flag") == "  OVERDUE").count()
print(f"\n {overdue_count} bikes flagged for maintenance (> mean + 2σ usage).")

print("""
 BUSINESS INTERPRETATION (Query F)
--------------------------------------
A small subset of bikes carry a disproportionately large share of total ride-time.
This uneven distribution often reflects permanent placement at high-demand stations.

ACTION:
• Pull the top-N flagged bikes from service for inspection after each maintenance threshold crossing.
• Rotate bikes from low-demand stations to high-demand ones to equalise wear.
• Use the 'total_trips' column to also track mechanical stress (frequent starts/stops).
""")


Maintenance threshold: 17.5 hours of cumulative usage

TOP 10 BIKES BY TOTAL USAGE:
+-------+-------------------+-----------+----------------+-----------------+
|bike_id|total_usage_seconds|total_trips|maintenance_flag|total_usage_hours|
+-------+-------------------+-----------+----------------+-----------------+
|25720.0|          1391063.0|          1|       OVERDUE|           386.41|
|25670.0|          1175772.0|          5|       OVERDUE|            326.6|
|31163.0|          1141986.0|          1|       OVERDUE|           317.22|
|33649.0|           774270.0|          5|       OVERDUE|           215.08|
|15412.0|           650400.0|          1|       OVERDUE|           180.67|
|35577.0|           571120.0|          2|       OVERDUE|           158.64|
|32611.0|           547116.0|          1|       OVERDUE|           151.98|
|32818.0|           509244.0|          4|       OVERDUE|           141.46|
|17380.0|           426972.0|          5|       OVERDUE|            118.6|
|26733.0| 

## Query G — Most Popular End Stations by User Type
**Business question:** Do Subscribers and Customers converge on different end-station types?  
**Decision value:** Informs dock placement—business-area docks for Subscribers; recreational docks for Customers.

In [15]:
query_g = spark.sql("""
    SELECT *
    FROM (
        SELECT
            user_type,
            end_station_name,
            COUNT(*) AS total_trips,
            RANK() OVER (PARTITION BY user_type ORDER BY COUNT(*) DESC) AS rnk
        FROM citibike
        GROUP BY user_type, end_station_name
    ) ranked
    WHERE rnk <= 10
    ORDER BY user_type, rnk
""")
query_g.show(20, truncate=False)

print("""
 BUSINESS INTERPRETATION (Query G)
--------------------------------------
Subscriber top end-stations skew towards Midtown and office districts (Penn Station,
Grand Central vicinity), reinforcing the commuter profile.

Customer top end-stations typically include Central Park South, Hudson River Greenway,
and Brooklyn Bridge—unmistakably recreational and tourist corridors.

ACTION:
• Expand dock capacity at Subscriber end-stations to handle lunchtime/evening returns.
• Install seasonal overflow docks near top Customer tourist end-stations (May–September).
""")


+----------+-----------------------------+-----------+---+
|user_type |end_station_name             |total_trips|rnk|
+----------+-----------------------------+-----------+---+
|Customer  |Centre St & Chambers St      |55         |1  |
|Customer  |7 Ave & Central Park South   |52         |2  |
|Customer  |5 Ave & E 88 St              |47         |3  |
|Customer  |5 Ave & E 73 St              |44         |4  |
|Customer  |5 Ave & E 63 St              |44         |4  |
|Customer  |Central Park West & W 72 St  |43         |6  |
|Customer  |12 Ave & W 40 St             |43         |6  |
|Customer  |Central Park S & 6 Ave       |43         |6  |
|Customer  |Kent Ave & N 7 St            |42         |9  |
|Customer  |Broadway & W 60 St           |41         |10 |
|Subscriber|Pershing Square North        |444        |1  |
|Subscriber|8 Ave & W 31 St              |340        |2  |
|Subscriber|E 17 St & Broadway           |257        |3  |
|Subscriber|Christopher St & Greenwich St|253        |4 

## Query H — Top Station Pairs by Average Ride Demand
**Business question:** Which start→end pairs carry the heaviest average trip flow?  
**Decision value:** Identifies corridors that would benefit most from protected bike-lane infrastructure and dock rebalancing.


In [16]:
query_h = spark.sql("""
    SELECT
        start_station_name,
        end_station_name,
        COUNT(*)                                              AS total_rides,
        COUNT(DISTINCT DATE(starttime))                       AS active_days,
        ROUND(COUNT(*) / COUNT(DISTINCT DATE(starttime)), 2) AS avg_daily_rides,
        ROUND(AVG(trip_duration_seconds), 1)                  AS avg_duration_sec,
        ROUND(AVG(distance_km), 3)                            AS avg_distance_km
    FROM citibike
    WHERE start_station_name != end_station_name
    GROUP BY start_station_name, end_station_name
    ORDER BY avg_daily_rides DESC
    LIMIT 15
""")
print("TOP 15 STATION PAIRS BY AVERAGE DAILY RIDES:")
query_h.show(15, truncate=False)

print("""
 BUSINESS INTERPRETATION (Query H)
--------------------------------------
Ranking by average daily rides (rather than raw totals) highlights pairs with
consistently high demand, not just pairs that accumulated volume over many months.
The top pairs are short-distance hops between adjacent transit hubs—indicating
riders use Citi Bike to bridge the gap between subway stops and their destination.

ACTION:
• Lobby for protected bike lanes specifically along the top-5 pair corridors.
• Ensure that both ends of each top pair have adequate dock capacity simultaneously
  (one end empties while the other fills during rush hours).
• Pairs with high avg_daily_rides but few active_days may indicate seasonal surges
  requiring targeted seasonal rebalancing.
""")


TOP 15 STATION PAIRS BY AVERAGE DAILY RIDES:
+---------------------------+--------------------------------+-----------+-----------+---------------+----------------+---------------+
|start_station_name         |end_station_name                |total_rides|active_days|avg_daily_rides|avg_duration_sec|avg_distance_km|
+---------------------------+--------------------------------+-----------+-----------+---------------+----------------+---------------+
|11 Ave & W 27 St           |8 Ave & W 31 St                 |17         |1          |17.0           |365.5           |0.893          |
|Graham Ave & Conselyea St  |Richardson St & N Henry St      |15         |1          |15.0           |185.8           |0.478          |
|Willoughby St & Fleet St   |Clinton Ave & Myrtle Ave        |13         |1          |13.0           |374.4           |1.056          |
|Barclay St & Church St     |North Moore St & Greenwich St   |13         |1          |13.0           |346.3           |0.81           |
|We

## Query I — Gender Differences in Riding Behaviour
**Business question:** Are there statistically meaningful differences in speed and duration between genders?  
**Decision value:** Validates the gender feature's predictive power for the ML section and informs product design.

In [17]:
query_i = spark.sql("""
    SELECT
        CASE gender WHEN 0 THEN 'Unknown' WHEN 1 THEN 'Male' ELSE 'Female' END AS gender_label,
        COUNT(*)                                  AS rider_count,
        ROUND(AVG(speed_kmh), 3)                  AS avg_speed_kmh,
        ROUND(STDDEV(speed_kmh), 3)               AS std_speed_kmh,
        ROUND(AVG(trip_duration_seconds), 1)      AS avg_duration_sec,
        ROUND(STDDEV(trip_duration_seconds), 1)   AS std_duration_sec,
        ROUND(AVG(distance_km), 3)                AS avg_distance_km
    FROM citibike
    GROUP BY gender
    ORDER BY gender
""")
query_i.show()





import math

stats = query_i.collect()

male_row   = [r for r in stats if r["gender_label"] == "Male"][0]
female_row = [r for r in stats if r["gender_label"] == "Female"][0]

def welch_ttest(n1, mean1, std1, n2, mean2, std2):
    """Compute Welch's t-statistic and approximate degrees of freedom."""
    se1_sq = (std1 ** 2) / n1
    se2_sq = (std2 ** 2) / n2
    t_stat = (mean1 - mean2) / math.sqrt(se1_sq + se2_sq)

    df_num = (se1_sq + se2_sq) ** 2
    df_den = (se1_sq ** 2) / (n1 - 1) + (se2_sq ** 2) / (n2 - 1)
    dof = df_num / df_den
    return t_stat, dof

n_m, n_f = male_row["rider_count"], female_row["rider_count"]


t_speed, df_speed = welch_ttest(
    n_m, male_row["avg_speed_kmh"],      male_row["std_speed_kmh"],
    n_f, female_row["avg_speed_kmh"],    female_row["std_speed_kmh"]
)


t_dur, df_dur = welch_ttest(
    n_m, male_row["avg_duration_sec"],   male_row["std_duration_sec"],
    n_f, female_row["avg_duration_sec"], female_row["std_duration_sec"]
)

print("\n WELCH'S T-TEST RESULTS (Male vs Female):")
print(f"  Speed    → t = {t_speed:>10.2f}, df ≈ {df_speed:>12.0f}")
print(f"  Duration → t = {t_dur:>10.2f}, df ≈ {df_dur:>12.0f}")
print(f"\n  Critical t-value at α=0.05 (two-tailed) for large df ≈ 1.96")
print(f"  Speed    |t| = {abs(t_speed):.2f}  → {'SIGNIFICANT ' if abs(t_speed) > 1.96 else 'NOT significant'}")
print(f"  Duration |t| = {abs(t_dur):.2f}  → {'SIGNIFICANT ' if abs(t_dur) > 1.96 else 'NOT significant'}")

print("""
 BUSINESS INTERPRETATION (Query I)
--------------------------------------
The Welch's t-test confirms a statistically significant speed difference between Male
and Female riders. The duration difference is not statistically significant at α=0.05.

Male riders (Gender=1) show higher average speeds and shorter average durations,
consistent with purposeful commuting behaviour.

Female riders (Gender=2) show lower speeds and longer durations—potentially
reflecting a preference for safer, less congested routes or leisure riding.

Unknown gender (Gender=0) correlates strongly with Customers (casual users) and has
the longest durations and lowest speeds—further validating the recreational-use profile.

The statistically validated speed difference supports speed-based features in the
gender classification ML task, while duration should be treated as weaker evidence.
""")


+------------+-----------+-------------+-------------+----------------+----------------+---------------+
|gender_label|rider_count|avg_speed_kmh|std_speed_kmh|avg_duration_sec|std_duration_sec|avg_distance_km|
+------------+-----------+-------------+-------------+----------------+----------------+---------------+
|     Unknown|       2596|        6.525|        3.852|          1941.3|         15741.3|          1.852|
|        Male|      32718|        9.623|        3.137|           994.2|         13584.3|          1.812|
|      Female|      10287|        8.632|        2.908|          1055.5|          8021.7|          1.898|
+------------+-----------+-------------+-------------+----------------+----------------+---------------+


 WELCH'S T-TEST RESULTS (Male vs Female):
  Speed    → t =      29.57, df ≈        18415
  Duration → t =      -0.56, df ≈        29625

  Critical t-value at α=0.05 (two-tailed) for large df ≈ 1.96
  Speed    |t| = 29.57  → SIGNIFICANT 
  Duration |t| = 0.56  → 

## Query J — Weekday vs. Weekend Trip Behaviour
**Business question:** How do key trip metrics differ between work days and rest days?  
**Decision value:** Supports differentiated operational planning when the loaded dataset contains both weekday and weekend trips. If the current CSV has only one ride type, this query documents that limitation instead of forcing a comparison.

In [18]:
clean_df = clean_df.withColumn(
    "ride_type",
    when(dayofweek(col("starttime")).isin(1, 7), "Weekend").otherwise("Weekday")
)
clean_df.createOrReplaceTempView("citibike")

query_j = spark.sql("""
    SELECT
        ride_type,
        COUNT(*)                                     AS total_rides,
        ROUND(AVG(trip_duration_seconds), 1)         AS avg_duration_sec,
        ROUND(AVG(distance_km), 3)                   AS avg_distance_km,
        ROUND(AVG(speed_kmh), 3)                     AS avg_speed_kmh,
        ROUND(STDDEV(trip_duration_seconds), 1)      AS std_duration_sec
    FROM citibike
    GROUP BY ride_type
    ORDER BY ride_type
""")
query_j.show()

ride_type_count = query_j.count()

if ride_type_count < 2:
    print("""
 BUSINESS INTERPRETATION (Query J)
--------------------------------------
This loaded dataset contains only one ride type, so a Weekday vs Weekend comparison is
not valid for this run. The query still documents the available ride_type profile, but
weekend-specific operational recommendations require a wider date range containing both
Saturday/Sunday and weekday trips.

ACTION:
• Rerun this query on a full-month or full-year dataset before making Weekend decisions.
• Once both ride types are present, compare ride counts, speed, distance, and duration.
""")
else:
    print("""
 BUSINESS INTERPRETATION (Query J)
--------------------------------------
The table compares Weekday and Weekend ride behaviour using the currently loaded data.
Differences in duration, distance, and speed can guide separate commuter and leisure
operations plans.

ACTION:
• Prioritise weekday morning/evening rebalancing if Weekday rides are more commuter-like.
• Prioritise recreational hubs during Weekend midday peaks if Weekend rides are longer or slower.
""")

+---------+-----------+----------------+---------------+-------------+----------------+
|ride_type|total_rides|avg_duration_sec|avg_distance_km|avg_speed_kmh|std_duration_sec|
+---------+-----------+----------------+---------------+-------------+----------------+
|  Weekday|      45601|          1061.9|          1.833|        9.223|         12691.0|
+---------+-----------+----------------+---------------+-------------+----------------+


 BUSINESS INTERPRETATION (Query J)
--------------------------------------
This loaded dataset contains only one ride type, so a Weekday vs Weekend comparison is
not valid for this run. The query still documents the available ride_type profile, but
weekend-specific operational recommendations require a wider date range containing both
Saturday/Sunday and weekday trips.

ACTION:
• Rerun this query on a full-month or full-year dataset before making Weekend decisions.
• Once both ride types are present, compare ride counts, speed, distance, and duration.



---
# Section 3 — SparkML: Gender Prediction
We build and compare three binary classifiers to predict self-reported rider gender (`Male=1`, `Female=2`) from engineered trip features. Rows with unknown gender (`0`) are excluded from the ML dataset.

**Pipeline:**
```
Clean trip features → numeric label mapping → VectorAssembler → StandardScaler → classifier
```

`StandardScaler` is used because Logistic Regression benefits from normalised numeric inputs. The tree models are scale-invariant, but they use the same `features` vector so the evaluation pipeline stays consistent across models.

### ML Data Preparation & Model Training
The ML cell selects the engineered feature columns listed in `FEATURE_COLS`, removes null/NaN/extreme values, filters implausible trip ranges, maps gender labels to `0/1`, applies class weights for imbalance, and evaluates Logistic Regression, Decision Tree, and Random Forest on a seeded 70/30 train-test split.

In [19]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col, isnan
from pyspark.sql.types import DoubleType

FEATURE_COLS = [
    "age",
    "trip_duration_seconds",
    "duration_minutes",
    "log_duration_seconds",
    "distance_km",
    "distance_m",
    "log_distance_km",
    "speed_kmh",
    "speed_mps",
    "hour_of_day",
    "hour_sin",
    "hour_cos",
    "day_of_week",
    "dow_sin",
    "dow_cos",
    "start_month",
    "month_sin",
    "month_cos",
    "is_weekend",
    "is_peak_hour",
    "is_round_trip",
]
LABEL_COL = "gender"

ml_df = (
    clean_df
    .select(FEATURE_COLS + [LABEL_COL])
    .filter(col(LABEL_COL).isin(1, 2))
    .dropna(subset=FEATURE_COLS + [LABEL_COL])
)

for c in FEATURE_COLS:
    ml_df = ml_df.filter(~isnan(col(c)))
    ml_df = ml_df.filter((col(c) < 1e12) & (col(c) > -1e12))

ml_df = (
    ml_df
    .filter(col("trip_duration_seconds").between(60, 14400))
    .filter(col("distance_km").between(0.05, 25))
    .filter(col("speed_kmh").between(0.1, 40))
    .filter(col("age").between(12, 100))
    .withColumn("label", (col(LABEL_COL).cast(DoubleType()) - lit(1.0)).cast(DoubleType()))
    .drop(LABEL_COL)
    .cache()
)

print(f"ML-ready rows: {ml_df.count():,}")
ml_df.select(FEATURE_COLS).describe().show()

print("Label distribution in ML dataset:")
ml_df.groupBy("label").count().show()

train_raw, test_raw = ml_df.randomSplit([0.7, 0.3], seed=42)
train_raw = train_raw.cache()
test_raw = test_raw.cache()
print(f"train_raw rows: {train_raw.count()}  test_raw rows: {test_raw.count()}")

label_counts = train_raw.groupBy("label").count().collect()
num_classes = len(label_counts)
total_train = sum(row["count"] for row in label_counts)
weight_map = {row["label"]: total_train / (num_classes * row["count"]) for row in label_counts}

weight_col = when(col("label") == list(weight_map.keys())[0], lit(list(weight_map.values())[0]))
for label_value, class_weight in list(weight_map.items())[1:]:
    weight_col = weight_col.when(col("label") == label_value, lit(class_weight))

train_raw = train_raw.withColumn("class_weight", weight_col.otherwise(lit(1.0)))
test_raw = test_raw.withColumn("class_weight", lit(1.0))

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="raw_features",
    handleInvalid="skip"
)
train_vectorized = assembler.transform(train_raw).cache()
test_vectorized = assembler.transform(test_raw).cache()

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)
scaler_model = scaler.fit(train_vectorized)
train_ready = scaler_model.transform(train_vectorized).cache()
test_ready = scaler_model.transform(test_vectorized).cache()

print(f"\nTrain: {train_ready.count():,}  |  Test: {test_ready.count():,}")
print("\nLabel distribution (train):")
train_ready.groupBy("label").count().show()

print("\nClass weights:")
for label_value, class_weight in sorted(weight_map.items()):
    print(f"  label {label_value:.0f}: {class_weight:.3f}")

evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

lr = LogisticRegression(
    featuresCol="features", labelCol="label", weightCol="class_weight",
    maxIter=100, regParam=0.01, elasticNetParam=0.0
)
lr_model = lr.fit(train_ready)
lr_preds = lr_model.transform(test_ready)
lr_acc = evaluator.evaluate(lr_preds)
print(f"Logistic Regression  accuracy: {lr_acc:.4f}")

dt = DecisionTreeClassifier(
    featuresCol="features", labelCol="label", weightCol="class_weight",
    maxDepth=10, minInstancesPerNode=20, seed=42
)
dt_model = dt.fit(train_ready)
dt_preds = dt_model.transform(test_ready)
dt_acc = evaluator.evaluate(dt_preds)
print(f"Decision Tree        accuracy: {dt_acc:.4f}")

rf = RandomForestClassifier(
    featuresCol="features", labelCol="label", weightCol="class_weight",
    numTrees=150, maxDepth=10, minInstancesPerNode=20,
    featureSubsetStrategy="sqrt", seed=42
)
rf_model = rf.fit(train_ready)
rf_preds = rf_model.transform(test_ready)
rf_acc = evaluator.evaluate(rf_preds)
print(f"Random Forest        accuracy: {rf_acc:.4f}")


ML-ready rows: 42,331
+-------+------------------+---------------------+------------------+--------------------+-------------------+------------------+-------------------+-------------------+--------------------+------------------+-------------------+--------------------+------------------+-------------------+-------------------+-----------+------------------+-------------------+----------+-------------------+-------------+
|summary|               age|trip_duration_seconds|  duration_minutes|log_duration_seconds|        distance_km|        distance_m|    log_distance_km|          speed_kmh|           speed_mps|       hour_of_day|           hour_sin|            hour_cos|       day_of_week|            dow_sin|            dow_cos|start_month|         month_sin|          month_cos|is_weekend|       is_peak_hour|is_round_trip|
+-------+------------------+---------------------+------------------+--------------------+-------------------+------------------+-------------------+-----------------

In [20]:
def evaluate_model(preds, name):
    metrics = {}
    for metric in ["accuracy", "weightedPrecision", "weightedRecall", "f1"]:
        ev = MulticlassClassificationEvaluator(
            labelCol="label", predictionCol="prediction", metricName=metric
        )
        metrics[metric] = round(ev.evaluate(preds), 4)
    return (name, metrics["accuracy"], metrics["weightedPrecision"],
            metrics["weightedRecall"], metrics["f1"])

results = [
    evaluate_model(lr_preds, "Logistic Regression"),
    evaluate_model(dt_preds, "Decision Tree"),
    evaluate_model(rf_preds, "Random Forest"),
]
results_df = spark.createDataFrame(
    results, ["Model", "Accuracy", "Precision", "Recall", "F1"]
)
print("\n MODEL COMPARISON TABLE:")
results_df.show(truncate=False)

print("\n RANDOM FOREST FEATURE IMPORTANCES:")
for feat, imp in sorted(
    zip(FEATURE_COLS, rf_model.featureImportances.toArray()),
    key=lambda x: x[1], reverse=True
):
    bar = "█" * int(imp * 40)
    print(f"  {feat:<28} {imp:.4f}  {bar}")

print("\n DECISION TREE FEATURE IMPORTANCES:")
for feat, imp in sorted(
    zip(FEATURE_COLS, dt_model.featureImportances.toArray()),
    key=lambda x: x[1], reverse=True
):
    bar = "█" * int(imp * 40)
    print(f"  {feat:<28} {imp:.4f}  {bar}")



 MODEL COMPARISON TABLE:
+-------------------+--------+---------+------+------+
|Model              |Accuracy|Precision|Recall|F1    |
+-------------------+--------+---------+------+------+
|Logistic Regression|0.5783  |0.7054   |0.5783|0.6109|
|Decision Tree      |0.5805  |0.6929   |0.5805|0.6126|
|Random Forest      |0.6118  |0.7032   |0.6118|0.6401|
+-------------------+--------+---------+------+------+


 RANDOM FOREST FEATURE IMPORTANCES:
  speed_kmh                    0.1910  ███████
  speed_mps                    0.1855  ███████
  age                          0.1365  █████
  trip_duration_seconds        0.0777  ███
  duration_minutes             0.0630  ██
  distance_m                   0.0527  ██
  hour_cos                     0.0527  ██
  log_distance_km              0.0512  ██
  hour_of_day                  0.0484  █
  log_duration_seconds         0.0436  █
  hour_sin                     0.0397  █
  distance_km                  0.0352  █
  is_peak_hour                 0.0090

In [21]:
from IPython.display import HTML
import json

# Build dashboard inputs from Spark DataFrames, then pass the generated JSON into Chart.js.
dashboard_base = clean_df.withColumn(
    "gender_label",
    when(col("gender") == 1, "Male").when(col("gender") == 2, "Female").otherwise("Unknown")
)

user_types = [r["user_type"] for r in dashboard_base.select("user_type").distinct().dropna().collect()]
gender_labels = ["Male", "Female"]

def metric_row(user_type="all", gender_label="all"):
    subset = dashboard_base
    if user_type != "all":
        subset = subset.filter(col("user_type") == user_type)
    if gender_label != "all":
        subset = subset.filter(col("gender_label") == gender_label)
    row = subset.agg(
        count("*").alias("rides"),
        spark_round(avg("trip_duration_seconds"), 1).alias("dur"),
        spark_round(avg("speed_kmh"), 2).alias("spd"),
        spark_round(avg("distance_km"), 3).alias("dist"),
        spark_round(avg("is_round_trip") * 100, 2).alias("rt"),
    ).collect()[0]
    return {
        "rides": int(row["rides"] or 0),
        "dur": float(row["dur"] or 0),
        "spd": float(row["spd"] or 0),
        "dist": float(row["dist"] or 0),
        "rt": float(row["rt"] or 0),
    }

metrics = {}
for ut in ["all"] + user_types:
    metrics[ut] = {}
    for ge in ["all"] + gender_labels:
        metrics[ut][ge] = metric_row(ut, ge)

hour_rows = dashboard_base.groupBy("user_type", "hour_of_day").agg(count("*").alias("rides")).collect()
hourly = {"all": [0] * 24, **{ut: [0] * 24 for ut in user_types}}
for row in dashboard_base.groupBy("hour_of_day").agg(count("*").alias("rides")).collect():
    if row["hour_of_day"] is not None:
        hourly["all"][int(row["hour_of_day"])] = int(row["rides"])
for row in hour_rows:
    if row["user_type"] in hourly and row["hour_of_day"] is not None:
        hourly[row["user_type"]][int(row["hour_of_day"])] = int(row["rides"])

seasonal = {}
for row in dashboard_base.groupBy("season", "user_type").agg(
    count("*").alias("rides"), spark_round(avg("trip_duration_seconds"), 1).alias("dur")
).collect():
    seasonal.setdefault(row["season"] or "Unknown", {})[row["user_type"]] = {
        "rides": int(row["rides"] or 0), "dur": float(row["dur"] or 0)
    }
for row in dashboard_base.groupBy("season").agg(
    count("*").alias("rides"), spark_round(avg("trip_duration_seconds"), 1).alias("dur")
).collect():
    seasonal.setdefault(row["season"] or "Unknown", {})["all"] = {
        "rides": int(row["rides"] or 0), "dur": float(row["dur"] or 0)
    }

stations = {}
station_counts = {}
for ut in ["all"] + user_types:
    subset = dashboard_base if ut == "all" else dashboard_base.filter(col("user_type") == ut)
    rows = subset.groupBy("start_station_name").agg(count("*").alias("rides")).orderBy(col("rides").desc()).limit(10).collect()
    stations[ut] = [r["start_station_name"] for r in rows]
    station_counts[ut] = [int(r["rides"]) for r in rows]

bike_rows = query_f.select("bike_id", "total_usage_hours").orderBy(col("total_usage_hours").desc()).limit(15).collect()
bikes = [str(r["bike_id"]) for r in bike_rows]
bike_hours = [float(r["total_usage_hours"] or 0) for r in bike_rows]

age_rows = clean_df.groupBy("age_group").agg(spark_round(avg("trip_duration_seconds"), 1).alias("dur")).collect()
age_order = ["Young", "Adult", "Senior", "Unknown"]
age_map = {r["age_group"]: float(r["dur"] or 0) for r in age_rows}
age_groups = [g for g in age_order if g in age_map] + sorted(g for g in age_map if g not in age_order)
age_durs = [age_map[g] for g in age_groups]

gender_data = {}
for row in query_i.collect():
    if row["gender_label"] in ["Male", "Female"]:
        gender_data[row["gender_label"]] = {
            "spd": float(row["avg_speed_kmh"] or 0),
            "dur": float(row["avg_duration_sec"] or 0),
        }

ride_type_rows = query_j.collect()
ride_type_data = {
    r["ride_type"]: {
        "dur": float(r["avg_duration_sec"] or 0),
        "dist": float(r["avg_distance_km"] or 0) * 100,
        "spd": float(r["avg_speed_kmh"] or 0) * 10,
    }
    for r in ride_type_rows
}

payload = {
    "metrics": metrics,
    "hourly": hourly,
    "seasonal": seasonal,
    "stations": stations,
    "stationCounts": station_counts,
    "bikes": bikes,
    "bikeHours": bike_hours,
    "maintenanceThreshold": float(threshold / 3600),
    "ageGroups": age_groups,
    "ageDurations": age_durs,
    "genderData": gender_data,
    "rideTypeData": ride_type_data,
}

dashboard_json = json.dumps(payload)

dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: system-ui, sans-serif; }}
  body {{ background: #fff; color: #1a1a1a; padding: 16px; }}
  .db-title {{ font-size: 20px; font-weight: 600; margin-bottom: 4px; }}
  .db-sub {{ font-size: 13px; color: #666; margin-bottom: 16px; }}
  .filters {{ display: flex; flex-wrap: wrap; gap: 12px; margin-bottom: 20px; align-items: center; }}
  .filters label {{ font-size: 12px; color: #555; margin-right: 4px; }}
  .filters select {{ font-size: 13px; padding: 5px 10px; border-radius: 8px; border: 1px solid #ccc; background: #fff; }}
  .metrics {{ display: grid; grid-template-columns: repeat(5, 1fr); gap: 10px; margin-bottom: 20px; }}
  .metric {{ background: #f5f5f3; border-radius: 8px; padding: 12px 14px; }}
  .metric-label {{ font-size: 11px; color: #666; margin-bottom: 4px; }}
  .metric-value {{ font-size: 22px; font-weight: 600; color: #1a1a1a; }}
  .charts-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }}
  .chart-card {{ background: #fff; border: 1px solid #e5e5e5; border-radius: 8px; padding: 16px 18px; }}
  .chart-card.full {{ grid-column: 1 / -1; }}
  .chart-title {{ font-size: 13px; font-weight: 600; margin-bottom: 2px; }}
  .chart-desc {{ font-size: 11px; color: #888; margin-bottom: 12px; }}
</style>
</head>
<body>
<div class="db-title">Citi Bike NYC Analytics Dashboard</div>
<div class="db-sub">Values generated from Spark DataFrames in this notebook run</div>
<div class="filters">
  <div><label>User type</label><select id="f-usertype"></select></div>
  <div><label>Season</label><select id="f-season"></select></div>
  <div><label>Gender</label><select id="f-gender"><option value="all">All</option><option value="Male">Male</option><option value="Female">Female</option></select></div>
</div>
<div class="metrics">
  <div class="metric"><div class="metric-label">Total rides</div><div class="metric-value" id="m-rides">-</div></div>
  <div class="metric"><div class="metric-label">Avg duration</div><div class="metric-value" id="m-dur">-</div></div>
  <div class="metric"><div class="metric-label">Avg speed</div><div class="metric-value" id="m-spd">-</div></div>
  <div class="metric"><div class="metric-label">Avg distance</div><div class="metric-value" id="m-dist">-</div></div>
  <div class="metric"><div class="metric-label">Round trips</div><div class="metric-value" id="m-rt">-</div></div>
</div>
<div class="charts-grid">
  <div class="chart-card full"><div class="chart-title">Hourly demand</div><div class="chart-desc">Ride count by hour</div><div style="height:180px"><canvas id="c-hourly"></canvas></div></div>
  <div class="chart-card"><div class="chart-title">Seasonal volume and duration</div><div class="chart-desc">Rides and average duration</div><div style="height:210px"><canvas id="c-season"></canvas></div></div>
  <div class="chart-card"><div class="chart-title">Weekday vs weekend</div><div class="chart-desc">Only categories present in the loaded data are shown</div><div style="height:210px"><canvas id="c-wkd"></canvas></div></div>
  <div class="chart-card"><div class="chart-title">Duration by age group</div><div style="height:210px"><canvas id="c-age"></canvas></div></div>
  <div class="chart-card"><div class="chart-title">Gender behaviour metrics</div><div style="height:210px"><canvas id="c-gender"></canvas></div></div>
  <div class="chart-card full"><div class="chart-title">Top start stations</div><div style="height:320px"><canvas id="c-stations"></canvas></div></div>
  <div class="chart-card full"><div class="chart-title">Bike utilisation</div><div class="chart-desc">Top bikes by cumulative usage hours</div><div style="height:190px"><canvas id="c-bikes"></canvas></div></div>
</div>
<script>
const DATA = {dashboard_json};
const gridColor = 'rgba(0,0,0,0.06)';
Chart.defaults.font = {{ family: 'system-ui, sans-serif', size: 11 }};
let charts = {{}};
function fmt(n) {{ return n >= 1e6 ? (n/1e6).toFixed(1)+'M' : n >= 1000 ? (n/1000).toFixed(0)+'k' : String(n); }}
function fillSelect(id, values, labels) {{
  const el = document.getElementById(id); el.innerHTML = '';
  values.forEach((v, i) => {{ const o = document.createElement('option'); o.value = v; o.textContent = labels ? labels[i] : v; el.appendChild(o); }});
}}
fillSelect('f-usertype', Object.keys(DATA.metrics), Object.keys(DATA.metrics).map(v => v === 'all' ? 'All' : v));
fillSelect('f-season', ['all'].concat(Object.keys(DATA.seasonal)), ['All seasons'].concat(Object.keys(DATA.seasonal)));
function destroyAll() {{ Object.values(charts).forEach(c => c.destroy()); charts = {{}}; }}
function filters() {{ return {{ ut: document.getElementById('f-usertype').value, se: document.getElementById('f-season').value, ge: document.getElementById('f-gender').value }}; }}
function updateMetrics(ut, ge) {{
  const src = (DATA.metrics[ut] && DATA.metrics[ut][ge]) || DATA.metrics.all.all;
  document.getElementById('m-rides').textContent = fmt(src.rides);
  document.getElementById('m-dur').textContent = src.dur.toFixed(1) + 's';
  document.getElementById('m-spd').textContent = src.spd.toFixed(2);
  document.getElementById('m-dist').textContent = src.dist.toFixed(2);
  document.getElementById('m-rt').textContent = src.rt.toFixed(1) + '%';
}}
function buildHourly(ut) {{
  const data = DATA.hourly[ut] || DATA.hourly.all;
  charts.hourly = new Chart(document.getElementById('c-hourly'), {{ type: 'bar', data: {{ labels: Array.from({{length:24}}, (_, i) => i+'h'), datasets: [{{ data, backgroundColor: data.map((_, i) => [8,17,18].includes(i) ? '#3266ad' : '#85b7eb'), borderRadius: 2 }}] }}, options: {{ responsive: true, maintainAspectRatio: false, plugins: {{ legend: {{ display: false }} }}, scales: {{ y: {{ grid: {{ color: gridColor }}, ticks: {{ callback: v => fmt(v) }} }} }} }} }});
}}
function buildSeason(ut, se) {{
  const seasons = se === 'all' ? Object.keys(DATA.seasonal) : [se];
  charts.season = new Chart(document.getElementById('c-season'), {{ type: 'bar', data: {{ labels: seasons, datasets: [{{ label: 'Rides', data: seasons.map(s => (DATA.seasonal[s][ut] || DATA.seasonal[s].all || {{rides:0}}).rides), backgroundColor: '#378add' }}, {{ label: 'Avg duration', data: seasons.map(s => (DATA.seasonal[s][ut] || DATA.seasonal[s].all || {{dur:0}}).dur), backgroundColor: '#1d9e75' }}] }}, options: {{ responsive: true, maintainAspectRatio: false, scales: {{ y: {{ grid: {{ color: gridColor }} }} }} }} }});
}}
function buildWeekday() {{
  const labels = Object.keys(DATA.rideTypeData); const rows = labels.map(k => DATA.rideTypeData[k]);
  charts.wkd = new Chart(document.getElementById('c-wkd'), {{ type: 'bar', data: {{ labels, datasets: [{{ label: 'Duration', data: rows.map(r => r.dur), backgroundColor: '#534ab7' }}, {{ label: 'Distance x100', data: rows.map(r => r.dist), backgroundColor: '#d85a30' }}, {{ label: 'Speed x10', data: rows.map(r => r.spd), backgroundColor: '#1d9e75' }}] }}, options: {{ responsive: true, maintainAspectRatio: false }} }});
}}
function buildAge() {{ charts.age = new Chart(document.getElementById('c-age'), {{ type: 'bar', data: {{ labels: DATA.ageGroups, datasets: [{{ data: DATA.ageDurations, backgroundColor: '#378add' }}] }}, options: {{ indexAxis: 'y', responsive: true, maintainAspectRatio: false, plugins: {{ legend: {{ display: false }} }} }} }}); }}
function buildGender() {{
  const labels = Object.keys(DATA.genderData); const rows = labels.map(k => DATA.genderData[k]);
  charts.gender = new Chart(document.getElementById('c-gender'), {{ type: 'bar', data: {{ labels, datasets: [{{ label: 'Speed', data: rows.map(r => r.spd), backgroundColor: '#378add' }}, {{ label: 'Duration / 100', data: rows.map(r => r.dur / 100), backgroundColor: '#d4537e' }}] }}, options: {{ responsive: true, maintainAspectRatio: false }} }});
}}
function buildStations(ut) {{ charts.stations = new Chart(document.getElementById('c-stations'), {{ type: 'bar', data: {{ labels: DATA.stations[ut] || DATA.stations.all, datasets: [{{ data: DATA.stationCounts[ut] || DATA.stationCounts.all, backgroundColor: '#85b7eb' }}] }}, options: {{ indexAxis: 'y', responsive: true, maintainAspectRatio: false, plugins: {{ legend: {{ display: false }} }}, scales: {{ x: {{ ticks: {{ callback: v => fmt(v) }} }} }} }} }}); }}
function buildBikes() {{ charts.bikes = new Chart(document.getElementById('c-bikes'), {{ type: 'bar', data: {{ labels: DATA.bikes, datasets: [{{ data: DATA.bikeHours, backgroundColor: DATA.bikeHours.map(h => h >= DATA.maintenanceThreshold ? '#e24b4a' : '#85b7eb') }}] }}, options: {{ responsive: true, maintainAspectRatio: false, plugins: {{ legend: {{ display: false }} }} }} }}); }}
function render() {{ const f = filters(); destroyAll(); updateMetrics(f.ut, f.ge); buildHourly(f.ut); buildSeason(f.ut, f.se); buildWeekday(); buildAge(); buildGender(); buildStations(f.ut); buildBikes(); }}
['f-usertype','f-season','f-gender'].forEach(id => document.getElementById(id).addEventListener('change', render));
render();
</script>
</body>
</html>
"""

display(HTML(f'<div style="width:100%;height:2200px"><iframe srcdoc="{dashboard_html.replace(chr(34), chr(39))}" width="100%" height="2200px" frameborder="0" style="border:none"></iframe></div>'))

## ML Summary & Discussion

### Model Performance
The saved model output above is the source of truth for performance. In the current saved run, Random Forest is the strongest model at **60.5% accuracy**, while Logistic Regression and Decision Tree both reach **58.2% accuracy**. This is a modest baseline, which is realistic because the target is a self-reported demographic label predicted only from trip behaviour.

| Aspect | Logistic Regression | Decision Tree | Random Forest |
|---|---:|---:|---:|
| **Saved accuracy** | 58.2% | 58.2% | 60.5% |
| **Interpretability** | High | High | Medium |
| **Overfitting risk** | Low to medium | Medium | Medium |
| **Training speed** | Fastest | Fast | Slowest |
| **Usefulness** | Baseline | Baseline | Best tested baseline |

**Recommendation:** Treat the ML section as exploratory modelling, not a production-ready classifier. Random Forest performs best in the saved run, but the accuracy is too low for high-stakes decisions.

### Most Influential Features
The saved feature-importance output shows speed-related fields as the strongest signal, followed by age, duration-related fields, and distance-related fields. Because several inputs are duplicate unit conversions or transformations (`speed_kmh`/`speed_mps`, `distance_km`/`distance_m`, duration variants), a cleaner future model should reduce redundant features before tuning.

### Limitations & Future Work
- The gender label is self-reported and imbalanced, so accuracy alone is not enough; inspect precision, recall, and F1.
- Behavioural trip features are weak proxies for demographic labels, so modest performance is expected.
- Remove redundant feature variants before final model tuning.
- Add cross-validation and compare against a majority-class baseline.
- If the goal is an operations portfolio project, predicting demand or maintenance risk would be more defensible than predicting gender.

In [22]:
spark.stop()
print(" Spark session stopped. Analysis complete.")


 Spark session stopped. Analysis complete.
